In [3]:
import re, sys
import polars as pl
from pathlib import Path
from pprint import pprint

import phonopy
from phonopy import alignment
from phonopy.strings import str_standardize

In [ ]:
# Pronunciation dictionary.
data_dir = Path.home() / "Languages/00Dictionaries/wikipron/data/scrape/tsv"
data_file = "ceb_latn_broad.tsv"

data_df = pl.read_csv(data_dir / data_file, separator="\t", has_header=False, new_columns=["orth", "phon"])
print(data_df.shape)
print(data_df.head())

(2953, 2)
shape: (5, 2)
┌──────────┬───────────────────┐
│ orth     ┆ phon              │
│ ---      ┆ ---               │
│ str      ┆ str               │
╞══════════╪═══════════════════╡
│ Abapo    ┆ ʔ a b a p o       │
│ Abayon   ┆ ʔ a b a j o n     │
│ Abello   ┆ ʔ a b e l j o     │
│ Abesamis ┆ ʔ a b e s a m i s │
│ Abril    ┆ ʔ a b ɾ i l       │
└──────────┴───────────────────┘


In [ ]:
aligner = alignment.SelfAligner()

alignments = []
for idx, x in enumerate(data_df["phon"].to_list()):
    x = str_standardize(x)
    try:
        align = aligner.get_alignment(x)
        align_best = align[0]
        align_str = aligner.markup_self_alignment( \
            x, align_best.alignment, align_best.offset)
        align_score = align_best[0]
        alignments.append((align_score, align_str))
    except:
        print(f"Error aligning {x}")
        alignments.append((None, None))
    if idx > 10:
        break

In [6]:
pprint(alignments)

[(0.4583333333333335, 'ʔ ⟨ a b ⟩ ⟨ a p ⟩ o'),
 (0.3750000000000001, '⟨ ʔ a ⟩ ⟨ b a ⟩ j o n'),
 (0.284241209316635, '⟨ ʔ a ⟩ ⟨ b e ⟩ l j o'),
 (0.8581050715824003, 'ʔ ⟨ a b e s ⟩ ⟨ a m i s ⟩'),
 (0.17307692307692324, 'ʔ a b ⟨ ɾ ⟩ i ⟨ l ⟩'),
 (0.49323591831134384, '⟨ ʔ a s e ⟩ ⟨ b e d o ⟩'),
 (0.4058216273156766, 'ʔ ⟨ a d ⟩ ⟨ a n ⟩'),
 (0.5731641549469864, '⟨ ʔ a d b e ⟩ n ⟨ t i s t a ⟩'),
 (0.4614493194451882, 'ʔ ⟨ a d b i ⟩ j ⟨ e n t o ⟩'),
 (0.4058216273156766, 'ʔ ⟨ a d ⟩ l a w ⟨ a n ⟩'),
 (0.4058216273156766, 'ʔ ⟨ a d ⟩ l a w ⟨ a n ⟩'),
 (0.25, '⟨ a ⟩ g ⟨ a ⟩ s')]
